# Amazon ML Challenge — Entity Resolution Pipeline

Idempotent, checkpointed. Every step writes to `OUT_DIR` on Drive and skips if
its output already exists. **Just run all cells** and let it work end-to-end.

Boot → normalize → block (symbolic + ANN) → candidates → features → train → predict.


## Step 0 — Boot

In [ ]:
import sys, os, importlib
sys.path.insert(0, '/content/drive/MyDrive/amazon_ml/repo')

import src._colab_boot as boot_mod
importlib.reload(boot_mod)
ctx = boot_mod.boot()

REPO_DIR    = ctx['REPO_DIR']
OUT_DIR     = ctx['OUT_DIR']
TRAIN_DIR   = ctx['TRAIN_DIR']
TEST_DIR    = ctx['TEST_DIR']
GPU         = ctx['gpu']
os.chdir(REPO_DIR)

## Step 1 — Normalize all 6 sources

Writes `*_clean.parquet`. Skips sources already normalized.


In [ ]:
import src.normalize as norm
importlib.reload(norm)
norm.normalize_sources(TRAIN_DIR, TEST_DIR, OUT_DIR)

## Step 2 — Symbolic blocking

Same-state prefix-4 + soundex on `name_norm`. CPU, fast.


In [ ]:
import src.blocking_symbolic as bsym
importlib.reload(bsym)
for split in ('train', 'test'):
    print(f"--- {split} ---")
    bsym.build_symbolic_candidates(OUT_DIR, split)

## Step 3 — ANN blocking (SKIPPED for faster CPU-only run)

ANN blocking is disabled. Using only symbolic candidates.
To re-enable: uncomment the cells below and ensure GPU runtime.


In [ ]:
print("ANN blocking SKIPPED - using symbolic candidates only")
print("To re-enable: change runtime to GPU, uncomment below, restart & run all")

# if not GPU:
#     print("WARNING: No GPU detected. This step will be 20x slower on CPU.")
#     print("Runtime -> Change runtime type -> T4 GPU, then restart and run all.")
#
# import src.blocking_ann as bann
# importlib.reload(bann)
# for split in ('train', 'test'):
#     print(f"--- {split} ---")
#     bann.build_ann_candidates(OUT_DIR, split, k=20)


## Step 4 — Union candidates


In [ ]:
import src.candidates as cand
importlib.reload(cand)
for split in ('train', 'test'):
    cand.union_candidates(OUT_DIR, split)

## Step 5 — Pairwise features


In [ ]:
import src.features as feat
importlib.reload(feat)
for split in ('train', 'test'):
    feat.build_features(OUT_DIR, split)

## Step 6 — Train LightGBM + tune threshold

Trains on train features + ground truth. Picks threshold maximizing set-F1
on a 15% held-out split of S1 entities.


In [ ]:
import src.train as trn
importlib.reload(trn)
trn.train(OUT_DIR, gt_parquet=f'{OUT_DIR}/train_ground_truth.parquet')

## Step 7 — Predict on test, write submission.tsv


In [ ]:
import src.predict as prd
importlib.reload(prd)
prd.predict(OUT_DIR)

## Step 8 — Validate submission format

Runs the official validator if present. Checks:
- Column headers match ground-truth format
- Every test S1 has exactly one row
- Matched ids are pipe-or-comma-separated per the spec


In [ ]:
import subprocess, os
validator = '/content/drive/MyDrive/amazon_ml/dataset/utils/validate_submission.py'
sub = f'{OUT_DIR}/submission.tsv'
if os.path.exists(validator):
    print(subprocess.run(
        ['python', validator, sub],
        capture_output=True, text=True).stdout)
else:
    print(f"validator not found at {validator} — skipping")

# Quick peek
import pandas as pd
df = pd.read_csv(sub, sep='\t')
print(f"submission.tsv: {len(df):,} rows")
print(df.head(5).to_string(index=False))